# Strands Agents with Local MCP Server — Classroom Ready Notebook

This notebook focuses only on **Model Context Protocol (MCP)** with Strands Agents.

The classroom environment may not allow calls to external MCP servers.  
So this notebook creates a **local MCP server** inside the notebook environment and connects a Strands agent to it using **Streamable HTTP**.

## What this notebook covers
- What MCP is and why it is useful
- How to create a local MCP server for classroom use
- How to expose MCP tools over a local HTTP endpoint
- How to connect Strands `MCPClient` to the local MCP server
- How to let a Strands agent call MCP tools
- How to inspect and test MCP tools
- Classroom activities for learners

## What this notebook does not cover
- external MCP servers - reason ADP blocks it



## Install packages

Run this cell only if the required packages are not already installed in your environment.

The notebook uses:
- `strands-agents` for the agent
- `mcp` for the local MCP server and client protocol
- `boto3` for Amazon Bedrock model access


In [ ]:
# Uncomment and run only if needed
!pip install -U strands-agents mcp fastapi uvicorn boto3

## Imports

The below code imports the required packages.

We use:
- `Agent` from Strands
- `MCPClient` to connect Strands to an MCP server
- `streamablehttp_client` because the local MCP server exposes an HTTP MCP endpoint
- `BedrockModel` to use Amazon Bedrock as the model provider


In [ ]:
import json
import os
import signal
import subprocess
import time
from pathlib import Path

import boto3

from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client

## Bedrock model configuration

Update the model ID if your classroom account uses a different Bedrock model.

Common classroom-friendly Nova model IDs:
- `amazon.nova-lite-v1:0`
- `amazon.nova-micro-v1:0`
- `amazon.nova-pro-v1:0`


In [ ]:
AWS_REGION = boto3.session.Session().region_name or "us-east-1"
MODEL_ID = "amazon.nova-lite-v1:0"

bedrock_model = BedrockModel(
    model_id=MODEL_ID,
    region_name=AWS_REGION,
    temperature=0.1,
)

print("Region:", AWS_REGION)
print("Model ID:", MODEL_ID)

## What MCP means in this notebook

MCP separates the **agent brain** from the **tools/actions**.

In this notebook:

| Component | Role |
|---|---|
| Strands Agent | Understands the user request and decides which tool to call |
| Local MCP Server | Hosts business tools such as leave application and leave summary |
| MCP Client | Connects the Strands agent to the MCP server |
| JSON file | Stores small demo state for leave requests |

This is useful for teaching because learners can see that the agent does not directly own the business logic.  
The business capability is exposed separately through MCP tools.


## Create a small local data store

The MCP server runs as a separate Python process.  
For this reason, it should not depend on notebook memory.

The below code creates a small JSON file that the MCP server will use as its local data store.


In [ ]:
BASE_DIR = Path.cwd() / "mcp_classroom_demo"
BASE_DIR.mkdir(exist_ok=True)

LEAVE_DB_PATH = BASE_DIR / "leave_requests.json"

if not LEAVE_DB_PATH.exists():
    LEAVE_DB_PATH.write_text("[]", encoding="utf-8")

print("Demo folder:", BASE_DIR)
print("Leave DB path:", LEAVE_DB_PATH)

## Create the local MCP server script

The below code writes a Python script that defines the MCP server.

The server exposes three MCP tools:

| MCP tool | Purpose |
|---|---|
| `apply_leave` | Applies leave for an employee |
| `get_leave_summary` | Shows leave history and total leave days |
| `get_leave_balance` | Shows remaining leave balance |

The server is local to the notebook environment. It does not call any external MCP website.


In [ ]:
SERVER_SCRIPT = BASE_DIR / "hr_leave_mcp_server.py"

mcp_server_code = f"""
import json
from pathlib import Path
from mcp.server.fastmcp import FastMCP

LEAVE_DB_PATH = Path(r"{str(LEAVE_DB_PATH)}")
ANNUAL_LEAVE_ALLOWANCE = 20

mcp = FastMCP("HR Leave MCP Server")


def load_requests():
    if not LEAVE_DB_PATH.exists():
        return []
    return json.loads(LEAVE_DB_PATH.read_text(encoding="utf-8"))


def save_requests(data):
    LEAVE_DB_PATH.write_text(json.dumps(data, indent=2), encoding="utf-8")


@mcp.tool(description="Apply leave for an employee. Use this when the user wants to apply or book leave.")
def apply_leave(employee_id: str = "EMP001", days: int = 1, reason: str = "Not specified") -> str:
    if days <= 0:
        return "Leave days must be a positive number."

    requests = load_requests()

    used_days = sum(
        int(record.get("days", 0))
        for record in requests
        if record.get("employee_id") == employee_id
    )

    if used_days + days > ANNUAL_LEAVE_ALLOWANCE:
        remaining = ANNUAL_LEAVE_ALLOWANCE - used_days
        return (
            f"Leave not applied. Employee {{employee_id}} has only {{remaining}} "
            f"day(s) remaining, but requested {{days}} day(s)."
        )

    record = {{
        "employee_id": employee_id,
        "days": int(days),
        "reason": reason
    }}

    requests.append(record)
    save_requests(requests)

    return f"Leave applied successfully for {{days}} day(s) for employee {{employee_id}}. Reason: {{reason}}"


@mcp.tool(description="Get leave summary and leave history for an employee.")
def get_leave_summary(employee_id: str = "EMP001") -> str:
    requests = load_requests()
    employee_records = [
        record for record in requests
        if record.get("employee_id") == employee_id
    ]

    if not employee_records:
        return f"No leave applications found for employee {{employee_id}}."

    total_days = sum(int(record.get("days", 0)) for record in employee_records)

    lines = [
        f"Employee {{employee_id}} has applied for a total of {{total_days}} day(s) of leave.",
        "Leave history:"
    ]

    for index, record in enumerate(employee_records, start=1):
        lines.append(
            f"{{index}}. {{record.get('days')}} day(s) - Reason: {{record.get('reason')}}"
        )

    return "\\n".join(lines)


@mcp.tool(description="Get remaining leave balance for an employee.")
def get_leave_balance(employee_id: str = "EMP001") -> str:
    requests = load_requests()
    used_days = sum(
        int(record.get("days", 0))
        for record in requests
        if record.get("employee_id") == employee_id
    )
    remaining = ANNUAL_LEAVE_ALLOWANCE - used_days

    return (
        f"Employee {{employee_id}} has used {{used_days}} day(s). "
        f"Remaining leave balance: {{remaining}} day(s)."
    )


if __name__ == "__main__":
    mcp.run(transport="streamable-http")
"""

SERVER_SCRIPT.write_text(mcp_server_code, encoding="utf-8")
print("MCP server script written to:", SERVER_SCRIPT)

## Start the local MCP server

The below code starts the MCP server as a background process.

It also stops any older server process from a previous notebook run.


In [ ]:
SERVER_LOG = BASE_DIR / "hr_leave_mcp_server.log"
SERVER_PID_FILE = BASE_DIR / "hr_leave_mcp_server.pid"

if SERVER_PID_FILE.exists():
    old_pid = int(SERVER_PID_FILE.read_text().strip())
    try:
        os.kill(old_pid, signal.SIGTERM)
        print(f"Stopped old MCP server process: {old_pid}")
        time.sleep(1)
    except ProcessLookupError:
        print(f"Old MCP server process was not running: {old_pid}")
    except Exception as error:
        print(f"Could not stop old MCP server process {old_pid}: {error}")

log_file = open(SERVER_LOG, "w", encoding="utf-8")

process = subprocess.Popen(
    ["python", str(SERVER_SCRIPT)],
    stdout=log_file,
    stderr=log_file,
    start_new_session=True,
)

SERVER_PID_FILE.write_text(str(process.pid), encoding="utf-8")
time.sleep(3)

print("MCP server started")
print("PID:", process.pid)
print("Log file:", SERVER_LOG)

## Check whether the MCP server is running

The below code checks the process ID and prints the latest server log lines.

If there is an error, check whether the `mcp` package is installed correctly.


In [ ]:
server_pid = int(SERVER_PID_FILE.read_text().strip())

try:
    os.kill(server_pid, 0)
    print(f"MCP server process {server_pid} is running")
except OSError:
    print(f"MCP server process {server_pid} is not running")

print("\nLatest log output:")
if SERVER_LOG.exists():
    print(SERVER_LOG.read_text(encoding="utf-8")[-2000:])
else:
    print("No log file found yet.")

## Connect to the MCP server from Strands

The local server exposes a Streamable HTTP MCP endpoint.

The default local endpoint is:

`http://127.0.0.1:8000/mcp/`

The below code creates an MCP client that Strands can use.


In [ ]:
MCP_SERVER_URL = "http://127.0.0.1:8000/mcp/"

mcp_client = MCPClient(
    lambda: streamablehttp_client(MCP_SERVER_URL)
)

print("MCP client created:", MCP_SERVER_URL)

## Inspect MCP tools before using the agent

Before connecting the tools to an agent, it is useful to inspect what the MCP server exposes.

This helps learners understand tool discovery.


In [ ]:
with mcp_client:
    tools = mcp_client.list_tools_sync()

print("Tools discovered from MCP server:")
for tool_item in tools:
    tool_name = getattr(tool_item, "tool_name", None) or getattr(tool_item, "name", None) or str(tool_item)
    print("-", tool_name)

## Create a Strands agent with MCP tools

The agent receives the MCP client in its `tools` list.

The agent can now decide when to call:
- `apply_leave`
- `get_leave_summary`
- `get_leave_balance`


In [ ]:
hr_mcp_agent = Agent(
    model=bedrock_model,
    tools=[mcp_client],
    system_prompt="""
You are an HR leave assistant.

You can use MCP tools for leave-related actions.

Rules:
1. Use apply_leave when the user wants to apply, book, or request leave.
2. Use get_leave_summary when the user asks for leave history or total leave taken.
3. Use get_leave_balance when the user asks how many leave days are remaining.
4. If the user does not provide employee_id, use EMP001.
5. Keep the final answer short and clear.
""",
)

print("HR MCP agent is ready")

## Example: Apply leave using the MCP tool

The agent should recognize that this is an action request and call the MCP tool.


In [ ]:
response = hr_mcp_agent(
    "Apply 2 days leave for employee EMP001 because of personal work."
)

print(response)

## Example: Ask for leave summary

The agent should call the MCP summary tool and return the leave history.


In [ ]:
response = hr_mcp_agent(
    "Show leave history for employee EMP001."
)

print(response)

## Example: Ask for leave balance

The agent should call the MCP balance tool.


In [ ]:
response = hr_mcp_agent(
    "How many leave days are remaining for employee EMP001?"
)

print(response)

## Example: Multi-step MCP request

The user asks for an action and then a state check.

The agent may need to call more than one MCP tool.


In [ ]:
response = hr_mcp_agent(
    "Apply 1 day leave for employee EMP001 for family work, then tell me the remaining leave balance."
)

print(response)

## Classroom activity: Add one more MCP tool

### Goal
Add a new MCP tool called `cancel_leave`.

### Learner task
Modify the MCP server script so that `cancel_leave`:
- accepts `employee_id`
- cancels the most recent leave request for that employee
- saves the updated JSON file
- returns a clear confirmation message

### Test prompts
After restarting the MCP server, test:

```text
Cancel the latest leave request for employee EMP001.
```

```text
Show leave history for employee EMP001.
```


In [ ]:
# Activity starter cell
# 1. Open the generated server file:
print(SERVER_SCRIPT)

# 2. Add a new @mcp.tool function called cancel_leave
# 3. Restart the MCP server by rerunning the server start cell
# 4. Test using hr_mcp_agent

## Stop the MCP server

Run this cell at the end of class if you want to stop the background MCP server.


In [ ]:
if SERVER_PID_FILE.exists():
    pid = int(SERVER_PID_FILE.read_text().strip())
    try:
        os.kill(pid, signal.SIGTERM)
        print(f"Stopped MCP server process: {pid}")
    except ProcessLookupError:
        print(f"MCP server process already stopped: {pid}")
else:
    print("No MCP server PID file found.")